# Synthetic Dataset of Driver Telematics 

## Import packages

In [1]:
# Import libraries
import sklearn
import pandas as pd
from sklearn.model_selection import train_test_split
import statsmodels.api as sm
import statsmodels.formula.api as smf
import numpy as np
import matplotlib.pyplot as plt
from typing import Callable
import dice_ml
import time
from numpy.random import seed
seed(42)

# Fix seed
import random
random.seed(42)

In [2]:
import os
os.getcwd()

'/Users/agathefernandesmachado/Documents/PhD/counterfactuals_insurance/scripts_python'

## Import dataset

In [3]:
df_train = pd.read_csv("../out/df_train.csv")
df_test = pd.read_csv("../out/df_test.csv")

## Frequency model: GLM Poisson

In [4]:
# Import fitted model from R
glm_design_matrix_train = pd.read_csv("../../counterfactuals_insurance/out/glm_design_matrix_train.csv")
glm_coef = pd.read_csv("../../counterfactuals_insurance/out/glm_coefficients.csv")

In [5]:
def preprocess_glm(df):
    glm_df = pd.get_dummies(df_test, dtype=int)
    glm_df = glm_df.rename(
        columns={
            "Accel_14miles": "Accel_14miles1", 
            "Brake_14miles": "Brake_14miles1",
            "Insured_sex_Male": "Insured_sexMale",
            "Region_Urban": "RegionUrban",
            "Pct_drive_rush_both": "Pct_drive_rushboth",
            "Pct_drive_rush_mostly_am": "Pct_drive_rushmostly_am",
            "Pct_drive_rush_mostly_pm": "Pct_drive_rushmostly_pm"
            }
        )
    glm_df = glm_df[
        ['Insured_age', 'Insured_sexMale', 'Car_age', 'Credit_score', 'RegionUrban', 'Pct_drive_rushmostly_am',
        'Pct_drive_rushmostly_pm', 'Pct_drive_rushboth', 'Accel_14miles1', 'Brake_14miles1', 'Y']
        ]
    return(glm_df)

In [6]:
def predict_glm_poisson(X, coef):
    df = X.copy()
    df["(Intercept)"] = 1
    beta = (
        coef
        .set_index("name")
        .reindex(df.columns)["value"]
        .values
        )  
    # Linear predictor
    eta = df.values @ beta #+ np.log(exposure)
    # Apply sigmoid
    rate = np.exp(eta)
    
    return np.array(rate) # Poisson mean

In [7]:
glm_df_test = preprocess_glm(df_test).drop(columns=['Y'])
glm_pred_test = predict_glm_poisson(glm_df_test, glm_coef)

In [8]:
class GLMWrapper:
    def __init__(self, coef):
        self.coef = coef

    def predict(self, X):
        return predict_glm_poisson(X, self.coef)

In [9]:
wrapped_model = GLMWrapper(glm_coef)
pred = wrapped_model.predict(glm_df_test)

In [10]:
glm_df_train = preprocess_glm(df_train)
# Continuous features except the outcome variable
continuous_features = glm_df_test.columns.tolist()

## DiCE

Documentation of the Python package: https://interpret.ml/DiCE/notebooks/DiCE_getting_started.html

In [11]:
# Define the DiCE data object
d = dice_ml.Data(dataframe = glm_df_train, continuous_features = continuous_features, outcome_name = 'Y')

In [12]:
# Create DiCE model object
m = dice_ml.Model(model=wrapped_model, backend='sklearn', model_type='regressor')  # use 'sklearn' trick for wrapper

In [13]:
telematics_variables = [
       'Pct_drive_rushmostly_am', 'Pct_drive_rushmostly_pm', 'Pct_drive_rushboth', 'Accel_14miles1', 'Brake_14miles1'
       ]

In [14]:
# Create explainer
genetic_explainer = dice_ml.Dice(d, m, method="genetic")
# Genetic method naturally handles categorical variables

In [15]:
indices = pd.read_csv("../../counterfactuals_insurance/out/sampled_indices.csv")
indices = indices.iloc[:, 0].to_list()
indices = [idx - 1 for idx in indices]

## Avec temps limite

In [16]:
import signal
import time


class TimeoutException(Exception):
    pass


def timeout_handler(signum, frame):
    raise TimeoutException


def generate_cf(idx_query):

    y_u = 0.9 * glm_pred_test[idx_query]
    query_instance_glm = glm_df_test.iloc[[idx_query]]

    start = time.time()

    # Programmer une interruption après 10 secondes
    signal.signal(signal.SIGALRM, timeout_handler)
    signal.alarm(10)

    try:
        cf = genetic_explainer.generate_counterfactuals(
            query_instance_glm,
            total_CFs=100,
            desired_range=[0, y_u],
            features_to_vary=telematics_variables,
            maxiterations=10
        )

        # Annuler l'alarme si DiCE termine avant 10 secondes
        signal.alarm(0)

        exec_time = time.time() - start

        print(f"idx {idx_query}: finished in {exec_time:.2f} seconds")

    except TimeoutException:

        signal.alarm(0)

        exec_time = time.time() - start

        print(
            f"idx {idx_query}: timeout after "
            f"{exec_time:.2f} seconds"
        )

        df_cf = pd.DataFrame(
            np.nan,
            index=[0],
            columns=[
                 "Insured_age",
                "Insured_sexMale",
                "Car_age",
                "Credit_score",
                "RegionUrban",
                *telematics_variables,
                "pred",
                "n_duplicates",
                "idx_query",
                "time"
            ]
        )

        df_cf["idx_query"] = idx_query + 1
        df_cf["time"] = exec_time

        return df_cf

    except Exception as e:

        signal.alarm(0)

        print(f"idx {idx_query}: error: {e}")

        return None

    # DiCE finished

    df_cf = cf.cf_examples_list[0].final_cfs_df

    cols_all = df_cf.columns.tolist()

    df_cf["n_duplicates"] = (
        df_cf.groupby(
            cols_all,
            dropna=False
        )[cols_all[0]].transform("size")
    )

    df_cf = (
        df_cf
        .drop_duplicates()
        .reset_index(drop=True)
    )

    cols = [
        "Pct_drive_rushmostly_am",
        "Pct_drive_rushmostly_pm",
        "Pct_drive_rushboth"
    ]

    df_cf = df_cf[
        df_cf[cols].sum(axis=1).isin([0, 1])
    ]

    df_cf = df_cf.rename(
        columns={"Y": "pred"}
    )

    exec_time = time.time() - start

    df_cf["idx_query"] = idx_query + 1
    df_cf["time"] = exec_time

    return df_cf

In [17]:
from joblib import Parallel, delayed, effective_n_jobs
effective_n_jobs(-1)

8

In [18]:
all_jobs = list(indices)

n_jobs = max(1, os.cpu_count() - 1)

dfs = Parallel(
    n_jobs=n_jobs,
    backend="loky",
    verbose=10
)(
    delayed(generate_cf)(idx)
    for idx in all_jobs
)

res_dice_test = pd.concat(dfs, ignore_index=True)
res_dice_test.to_csv("../out/cf_dice_test.csv", index=False)

[Parallel(n_jobs=7)]: Using backend LokyBackend with 7 concurrent workers.
  0%|          | 0/1 [00:00<?, ?it/s]

idx 6405: finished in 2.77 seconds
idx 3009: finished in 2.86 seconds


  0%|          | 0/1 [00:00<?, ?it/s]

idx 3049: finished in 2.79 seconds


100%|██████████| 1/1 [00:06<00:00,  6.38s/it]
[Parallel(n_jobs=7)]: Done   4 tasks      | elapsed:    8.2s
  0%|          | 0/1 [00:00<?, ?it/s]

...

idx 4357: timeout after 10.00 seconds


  0%|          | 0/1 [00:10<?, ?it/s]


idx 8915: timeout after 10.00 seconds


  0%|          | 0/1 [00:10<?, ?it/s]
[Parallel(n_jobs=7)]: Done 1000 out of 1000 | elapsed: 197.1min finished


idx 5322: timeout after 10.00 seconds
